# Day 2 — Calculus and Optimization

**Workshop:** Mathematical Foundations of Modern AI

Companion notebook to the Day 2 lecture notes. Five parts:

1. Hand-compute a gradient, verify it against PyTorch's autograd.
2. Add a manual backward pass to the NumPy MLP from Day 1.
3. Gradient-check: same input, same weights — manual gradients must match PyTorch's to the last decimal.
4. Train the manual MLP on MNIST with vanilla SGD. No frameworks.
5. Switch to PyTorch + Adam, then sweep the learning rate and look at the four characteristic learning-curve shapes.

---

## Part 1 — Gradients by hand and by autograd

Take the function
$$
f(x_1, x_2) = x_1^2 + 3 x_2^2.
$$
From the lecture notes, $\nabla f = (2 x_1, 6 x_2)$. At $(1, 1)$ the gradient should be $(2, 6)$. PyTorch's autograd will tell us the same thing.

In [ ]:
import torch

x = torch.tensor([1.0, 1.0], requires_grad=True)
f = x[0]**2 + 3 * x[1]**2
f.backward()
print(f"Autograd gradient at (1, 1): {x.grad.tolist()}")
print(f"Hand-derived gradient:       [2.0, 6.0]")

Same result. The point is not that autograd is right --- of course it is --- but that you can always check it against a hand calculation on a small example, which is how every new layer gets debugged.

---

## Part 2 — A manual backward pass for the Day 1 MLP

Implement the five lines of backprop from the lecture notes, for the same 2-layer MLP. This is the only piece of code in the entire workshop where we explicitly write the chain rule. After today, autograd handles it.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)

def init_mlp(d_in: int, d_hidden: int, d_out: int):
    W1 = rng.standard_normal((d_hidden, d_in)) * np.sqrt(2.0 / d_in)
    b1 = np.zeros(d_hidden)
    W2 = rng.standard_normal((d_out, d_hidden)) * np.sqrt(2.0 / d_hidden)
    b2 = np.zeros(d_out)
    return W1, b1, W2, b2

def softmax(z):
    z = z - z.max(axis=1, keepdims=True)   # numerical stability
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

def forward(x, params):
    W1, b1, W2, b2 = params
    h_pre = x @ W1.T + b1
    h = np.maximum(0, h_pre)
    y = h @ W2.T + b2
    return y, (x, h_pre, h)

def cross_entropy_loss(logits, y_true):
    """Cross-entropy with integer-class targets. Returns scalar loss."""
    p = softmax(logits)
    n = logits.shape[0]
    return -np.mean(np.log(p[np.arange(n), y_true] + 1e-12))

def backward(logits, y_true, cache, params):
    """Manual backprop. Returns gradients in the same order as params."""
    W1, b1, W2, b2 = params
    x, h_pre, h = cache
    n = logits.shape[0]

    # One-hot of the labels
    y_onehot = np.zeros_like(logits)
    y_onehot[np.arange(n), y_true] = 1.0

    # Five lines of backprop:
    dlogits = (softmax(logits) - y_onehot) / n              # (n, k)
    dW2 = dlogits.T @ h                                     # (k, m)
    db2 = dlogits.sum(axis=0)                               # (k,)
    dh = dlogits @ W2                                       # (n, m)
    dh_pre = dh * (h_pre > 0)                               # ReLU mask
    dW1 = dh_pre.T @ x                                      # (m, d)
    db1 = dh_pre.sum(axis=0)                                # (m,)
    return dW1, db1, dW2, db2

---

## Part 3 — Gradient check against PyTorch autograd

Copy the same weights into a PyTorch MLP, feed the same input, and verify every gradient matches the manual computation to within numerical tolerance. If this check passes, the backward implementation is correct.

In [ ]:
import torch
import torch.nn as nn

# Small dimensions for the check
d_in, d_hidden, d_out = 20, 16, 5
n_batch = 8

# Initialize numpy MLP
params = init_mlp(d_in, d_hidden, d_out)
W1_np, b1_np, W2_np, b2_np = params

# Mirror to PyTorch with the *same* weights
torch_mlp = nn.Sequential(nn.Linear(d_in, d_hidden), nn.ReLU(), nn.Linear(d_hidden, d_out))
with torch.no_grad():
    torch_mlp[0].weight.copy_(torch.tensor(W1_np))
    torch_mlp[0].bias.copy_(torch.tensor(b1_np))
    torch_mlp[2].weight.copy_(torch.tensor(W2_np))
    torch_mlp[2].bias.copy_(torch.tensor(b2_np))

# Same input, same labels
x_np = rng.standard_normal((n_batch, d_in))
y_np = rng.integers(0, d_out, size=n_batch)

# Manual forward + backward
logits_np, cache = forward(x_np, params)
loss_np = cross_entropy_loss(logits_np, y_np)
dW1, db1, dW2, db2 = backward(logits_np, y_np, cache, params)

# PyTorch forward + backward
x_t = torch.tensor(x_np, dtype=torch.float64).float()
y_t = torch.tensor(y_np, dtype=torch.long)
logits_t = torch_mlp(x_t)
loss_t = nn.functional.cross_entropy(logits_t, y_t)
loss_t.backward()

print(f"Loss (manual):  {loss_np:.6f}")
print(f"Loss (autograd): {loss_t.item():.6f}")
print()
print("Gradient-check (max absolute difference):")
print(f"  W1: {np.max(np.abs(dW1 - torch_mlp[0].weight.grad.numpy())):.2e}")
print(f"  b1: {np.max(np.abs(db1 - torch_mlp[0].bias.grad.numpy())):.2e}")
print(f"  W2: {np.max(np.abs(dW2 - torch_mlp[2].weight.grad.numpy())):.2e}")
print(f"  b2: {np.max(np.abs(db2 - torch_mlp[2].bias.grad.numpy())):.2e}")

All four differences should be on the order of $10^{-7}$ or smaller --- the level of numerical noise from float32 arithmetic. The backward pass is correct.

---

## Part 4 — Train the manual MLP on MNIST with vanilla SGD

Use only NumPy. Our own training loop. The point is to see that 50 lines of NumPy is all it takes to train a network from scratch.

In [ ]:
from torchvision import datasets, transforms

# Load MNIST through torchvision but convert to NumPy arrays
transform = transforms.Compose([transforms.ToTensor(), transforms.Lambda(lambda x: x.view(-1))])
train_ds = datasets.MNIST(root=".", train=True, download=True, transform=transform)
test_ds  = datasets.MNIST(root=".", train=False, download=True, transform=transform)

X_train = np.stack([train_ds[i][0].numpy() for i in range(len(train_ds))])
y_train = np.array([train_ds[i][1] for i in range(len(train_ds))])
X_test  = np.stack([test_ds[i][0].numpy() for i in range(len(test_ds))])
y_test  = np.array([test_ds[i][1] for i in range(len(test_ds))])
print(f"Train: {X_train.shape}, {y_train.shape}")
print(f"Test:  {X_test.shape}, {y_test.shape}")

In [ ]:
import time

rng = np.random.default_rng(0)
params = init_mlp(d_in=784, d_hidden=256, d_out=10)
lr = 0.1
batch_size = 128
n_epochs = 3

def accuracy(params, X, y, batch_size=512):
    n = X.shape[0]
    correct = 0
    for start in range(0, n, batch_size):
        logits, _ = forward(X[start:start+batch_size], params)
        correct += (logits.argmax(axis=1) == y[start:start+batch_size]).sum()
    return correct / n

t0 = time.time()
history = {"loss": [], "test_acc": []}
for epoch in range(n_epochs):
    perm = rng.permutation(len(X_train))
    epoch_loss = 0.0
    n_batches = 0
    for start in range(0, len(X_train), batch_size):
        idx = perm[start:start+batch_size]
        x_batch, y_batch = X_train[idx], y_train[idx]

        logits, cache = forward(x_batch, params)
        loss = cross_entropy_loss(logits, y_batch)
        grads = backward(logits, y_batch, cache, params)

        params = tuple(p - lr * g for p, g in zip(params, grads))   # SGD step

        epoch_loss += loss
        n_batches += 1

    acc = accuracy(params, X_test, y_test)
    history["loss"].append(epoch_loss / n_batches)
    history["test_acc"].append(acc)
    print(f"Epoch {epoch+1}: mean train loss = {epoch_loss/n_batches:.4f}, test acc = {acc:.4f}  ({time.time()-t0:.1f}s)")

Vanilla SGD with lr=0.1 should reach ~96% test accuracy in three epochs. The same architecture, the same data, no framework training loop. Everything you see in PyTorch is wrapping this.

---

## Part 5 — PyTorch + Adam, and a learning-rate sweep

Switch back to PyTorch. Same architecture, Adam instead of vanilla SGD. Then sweep the learning rate over four orders of magnitude and observe the four learning-curve shapes from the lecture.

In [ ]:
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)

def make_mlp():
    return nn.Sequential(nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 10)).to(device)

def train_short(lr: float, n_steps: int = 300):
    """Train for n_steps mini-batches, return the per-batch loss curve."""
    torch.manual_seed(0)
    model = make_mlp()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    losses = []
    step = 0
    for x, y in train_loader:
        if step >= n_steps:
            break
        x, y = x.to(device), y.to(device)
        loss = loss_fn(model(x), y)
        opt.zero_grad()
        loss.backward()
        opt.step()
        losses.append(loss.item())
        step += 1
    return losses

learning_rates = [1e-1, 1e-2, 1e-3, 1e-4]
curves = {lr: train_short(lr) for lr in learning_rates}

fig, ax = plt.subplots(figsize=(9, 5))
for lr, losses in curves.items():
    ax.plot(losses, label=f"lr = {lr}")
ax.set_xlabel("Mini-batch step")
ax.set_ylabel("Training loss")
ax.set_yscale("log")
ax.set_title("Learning curves at four learning rates (Adam, 2-layer MLP, MNIST)")
ax.legend()
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()

What you should see:

- `lr = 1e-1`: loss spikes upward, then chaotic. Divergence.
- `lr = 1e-2`: noisy but decreasing. Workable but not great.
- `lr = 1e-3`: clean, fast decrease. The Adam default for a reason.
- `lr = 1e-4`: slow, smooth decrease. Underfitting on this time budget.

Three orders of magnitude in learning rate produces three qualitatively different outcomes on the same model and the same data. This is why hyperparameter choice is not optional --- and why the first thing to look at when training fails is the learning curve.

---

## What you have built

- A correctness check between hand-derived gradients and PyTorch autograd.
- A complete training loop for a 2-layer MLP, with backprop you wrote yourself.
- The same training in PyTorch with Adam, for comparison.
- A controlled experiment showing how learning rate determines the shape of training.

Day 3 reframes everything you have done in probabilistic terms. The cross-entropy loss you used today is, in fact, the negative log-likelihood of the labels under the model's predicted distribution. Once we accept that framing, generative models follow naturally.